# Session 7.3: Transfer Learning for Vision

**Learning Objectives:**
- Master transfer learning with pre-trained models
- Use VGG16 for Cats vs Dogs classification
- Achieve >90% accuracy with limited data
- Fine-tune pre-trained networks
- Understand when to use transfer learning

**Dataset:** Cats vs Dogs (subset: 4,000 images)
**Time Estimate:** 90-120 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os

np.random.seed(42)
tf.random.set_seed(42)
print(f"TensorFlow: {tf.__version__}")

## 1. Prepare Dataset

**Note:** Download Cats vs Dogs dataset from Kaggle or TensorFlow Datasets.
Expected directory structure:
```
data/
  train/
    cats/
    dogs/
  validation/
    cats/
    dogs/
```

In [ ]:
# Data generators with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

# Update paths to your data directory
train_dir = 'data/cats_vs_dogs/train'
val_dir = 'data/cats_vs_dogs/validation'

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Classes: {train_generator.class_indices}")

## 2. Load Pre-trained VGG16

VGG16 was trained on ImageNet (1.2M images, 1000 classes).
We'll use it as a feature extractor.

In [ ]:
# Load VGG16 without top layers
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model layers
base_model.trainable = False

print("VGG16 loaded successfully.")
print(f"Total layers: {len(base_model.layers)}")
print(f"Trainable: {base_model.trainable}")

## 3. Build Transfer Learning Model

In [ ]:
# Build model on top of VGG16
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. Phase 1: Train Classifier (Frozen Base)

In [ ]:
print("Phase 1: Training classifier with frozen VGG16...\n")

history_phase1 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[
        EarlyStopping(patience=5, restore_best_weights=True),
        ModelCheckpoint('best_model_phase1.h5', save_best_only=True)
    ]
)

phase1_acc = history_phase1.history['val_accuracy'][-1]
print(f"\nPhase 1 Validation Accuracy: {phase1_acc:.4f}")

## 5. Phase 2: Fine-tuning (Unfreeze Top Layers)

In [ ]:
print("Phase 2: Fine-tuning top layers...\n")

# Unfreeze last 4 layers of VGG16
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_phase2 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[
        EarlyStopping(patience=5, restore_best_weights=True),
        ModelCheckpoint('best_model_phase2.h5', save_best_only=True)
    ]
)

phase2_acc = history_phase2.history['val_accuracy'][-1]
print(f"\nPhase 2 Validation Accuracy: {phase2_acc:.4f}")
print(f"Improvement: {(phase2_acc - phase1_acc)*100:.2f}%")

## 6. Visualizations

In [ ]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Combine both phases
all_acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
all_val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']

axes[0].plot(all_acc, label='Training')
axes[0].plot(all_val_acc, label='Validation')
axes[0].axvline(x=len(history_phase1.history['accuracy']), color='r', 
                linestyle='--', label='Fine-tuning starts')
axes[0].set_title('Accuracy Over Both Phases', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

all_loss = history_phase1.history['loss'] + history_phase2.history['loss']
all_val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

axes[1].plot(all_loss, label='Training')
axes[1].plot(all_val_loss, label='Validation')
axes[1].axvline(x=len(history_phase1.history['loss']), color='r', 
                linestyle='--', label='Fine-tuning starts')
axes[1].set_title('Loss Over Both Phases', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

if phase2_acc > 0.90:
    print("\n✓ SUCCESS: Achieved >90% accuracy with transfer learning!")

## 7. Test Predictions

In [ ]:
# Visualize predictions on validation set
import matplotlib.pyplot as plt

# Get a batch from validation generator
images, labels = next(val_generator)

# Make predictions
predictions = model.predict(images)

# Display
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Sample Predictions: Cats vs Dogs', fontsize=16, fontweight='bold')

class_names = ['Cat', 'Dog']

for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        pred_class = 'Dog' if predictions[i] > 0.5 else 'Cat'
        true_class = class_names[int(labels[i])]
        conf = predictions[i][0] if predictions[i] > 0.5 else 1 - predictions[i][0]
        
        color = 'green' if pred_class == true_class else 'red'
        ax.set_title(f'True: {true_class}\nPred: {pred_class}\nConf: {conf:.2%}',
                     color=color, fontweight='bold', fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.show()

## 8. Save Final Model

In [ ]:
model.save('cats_vs_dogs_vgg16.h5')
print("Model saved successfully.")

## Key Takeaways

### Transfer Learning Benefits:
- **Less Data Required**: Achieved >90% with only 4,000 images
- **Faster Training**: Pre-trained features save time
- **Better Performance**: Leverages ImageNet knowledge
- **Two-Phase Strategy**: Feature extraction → Fine-tuning

### When to Use Transfer Learning:
- ✓ Limited training data
- ✓ Similar task to pre-training (natural images)
- ✓ Limited computational resources
- ✗ Very different domain (medical images, satellite imagery)

### Best Practices:
1. Start with frozen base model
2. Train classifier layers first
3. Fine-tune top layers with small learning rate
4. Use data augmentation
5. Monitor for overfitting

**Session Complete!**